# Bangla ASR test: `Noobbbbb/whisper-small-bn-ctc`

Encoder of a Bengali-fine-tuned **Whisper-small** + a small **CTC head** (the slow Whisper decoder
is removed). 102M params, ~205 MB, **Apache-2.0**.
Card: validation WER 33.8% / CER 8.0%; FLEURS-bn WER 38.1% / CER 11.0%. Output is Bangla
script only (no English words, no punctuation).

> **RTF** = processing time / audio duration. Below 1.0 is faster than real time.
> Use **Runtime > Change runtime type > CPU** to measure CPU-server speed.

## 1. Settings

In [ ]:
THREADS = 0               # CPU threads, 0 = all cores
CHUNK_SEC = 30            # Whisper works on 30 s windows; keep <= 30

AUDIO_SOURCE = "upload"   # "upload" or "drive"
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/sample-audio"

## 2. Install dependencies (~1 minute)

In [ ]:
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null
!pip install -q transformers safetensors huggingface_hub librosa soundfile jiwer

## 3. Imports and environment check

In [ ]:
import os, sys, time, json, shutil, traceback
from pathlib import Path
import numpy as np, torch, librosa
if THREADS:
    torch.set_num_threads(THREADS)
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cores  :", os.cpu_count(), "| torch threads:", torch.get_num_threads())
!lscpu | grep -E "Model name|^CPU\(s\)"
import transformers
print("transformers:", transformers.__version__)

## 4. Download the model (~205 MB)

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"   # Xet transfer can stall; plain HTTP is reliable
from huggingface_hub import snapshot_download
REPO_ID = "Noobbbbb/whisper-small-bn-ctc"
MODEL_DIR = Path("/content/model")
token = None

t0 = time.time()
snapshot_download(REPO_ID, local_dir=MODEL_DIR, token=token, allow_patterns=['*.json', '*.safetensors', 'README.md'])
print(f"Downloaded in {time.time()-t0:.0f}s")
!du -sh /content/model; ls -R /content/model | head -30

## 5. Runner (same code as `whisper_small_bn_ctc/whisper_ctc.py`)

In [ ]:
"""
Runner for Noobbbbb/whisper-small-bn-ctc: Whisper-small encoder + CTC head (no decoder).

Same logic as the repo's `ctc_inference.py` (Apache-2.0); kept as a local reviewed copy
so no downloaded code is executed. Loads from a local folder only (offline).

Pipeline: wav 16 kHz -> Whisper log-mel (padded to 30 s) -> Whisper encoder (12 layers)
          -> 2-layer transformer CTC head -> greedy CTC (61 symbols: blank, space, 59 Bangla chars)
"""

import json
import math
import re
import unicodedata
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SR = 16000
WINDOW_SEC = 30  # Whisper always works on 30-second windows


def norm_text(t: str) -> str:
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[^ঀ-৿ ]", " ", t)  # keep Bangla block + spaces only
    return re.sub(r"\s+", " ", t).strip()


class CTCHead(nn.Module):
    def __init__(self, dim, n_vocab, kind="transformer", n_layers=2, dropout=0.0):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)
        self.body = None
        if kind == "transformer":
            layer = nn.TransformerEncoderLayer(dim, 8, dim * 4, dropout, activation="gelu",
                                               batch_first=True, norm_first=True)
            self.body = nn.TransformerEncoder(layer, n_layers, norm=nn.LayerNorm(dim),
                                              enable_nested_tensor=False)
        self.out = nn.Linear(dim, n_vocab)

    def forward(self, x, lengths):
        x = self.drop(self.norm(x))
        if self.body is not None:
            pad = torch.arange(x.shape[1], device=x.device)[None, :] >= lengths[:, None]
            x = self.body(x, src_key_padding_mask=pad)
        return F.log_softmax(self.out(x), dim=-1)


class WhisperCTC:
    def __init__(self, model_dir, threads: int = 0):
        from safetensors.torch import load_file
        from transformers import WhisperConfig, WhisperFeatureExtractor
        from transformers.models.whisper.modeling_whisper import WhisperEncoder

        path = Path(model_dir)
        if not (path / "encoder.safetensors").exists():
            raise FileNotFoundError(f"encoder.safetensors not found in {path}. Run download_model.py")
        if threads:
            torch.set_num_threads(threads)

        self.cfg = json.loads((path / "ctc_config.json").read_text(encoding="utf-8"))
        self.vocab = json.loads((path / "vocab.json").read_text(encoding="utf-8"))

        enc = WhisperEncoder(WhisperConfig.from_json_file(str(path / "encoder_config.json")))
        if self.cfg.get("encoder_was_cut"):  # upper layers removed, and with them the final norm
            enc.layer_norm = nn.Identity()
        enc.load_state_dict(load_file(str(path / "encoder.safetensors")))
        self.encoder = enc.float().eval()

        self.head = CTCHead(self.cfg["dim"], len(self.vocab), self.cfg["head"], self.cfg["head_layers"])
        self.head.load_state_dict(load_file(str(path / "head.safetensors")))
        self.head = self.head.float().eval()
        self.fe = WhisperFeatureExtractor.from_pretrained(str(path), local_files_only=True)

    @torch.inference_mode()
    def transcribe(self, y: np.ndarray) -> str:
        """y: 16 kHz mono float array, at most 30 s (the caller chunks longer audio)."""
        y = np.asarray(y, dtype=np.float32)
        if len(y) < 400:
            return ""
        feats = self.fe([y], sampling_rate=SR, return_tensors="pt").input_features
        n = min(1500, math.ceil(len(y) / SR * 50))  # encoder frames that hold real audio
        h = self.encoder(feats).last_hidden_state[:, :n].float()
        ids = self.head(h, torch.tensor([n])).argmax(-1)[0].numpy()
        keep = ids[np.insert(ids[1:] != ids[:-1], 0, True)]  # merge repeats, then drop blanks
        return norm_text("".join(self.vocab[i] for i in keep if i != 0))

## 6. Load the model

In [ ]:
t0 = time.time()
model = WhisperCTC(MODEL_DIR, threads=THREADS)
LOAD_SEC = time.time() - t0
n = sum(p.numel() for p in model.encoder.parameters()) + sum(p.numel() for p in model.head.parameters())
print(f"Model loaded in {LOAD_SEC:.1f}s | params {n/1e6:.1f}M")

## 7. Get your audio
* **upload**: select audio files, plus optional reference transcripts named like the audio
  (`call.mp3` + `call.txt`) to get WER/CER.
* **drive**: mounts Google Drive and reads `DRIVE_AUDIO_DIR`.

In [ ]:
AUDIO_DIR = Path("/content/sample-audio")
AUDIO_DIR.mkdir(exist_ok=True)
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus", ".webm"}

if AUDIO_SOURCE == "upload":
    from google.colab import files
    for name, data in files.upload().items():
        (AUDIO_DIR / name).write_bytes(data)
elif AUDIO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for p in Path(DRIVE_AUDIO_DIR).iterdir():
        if p.suffix.lower() in AUDIO_EXTS | {".txt"}:
            shutil.copy(p, AUDIO_DIR / p.name)

AUDIO_FILES = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTS)
print(f"{len(AUDIO_FILES)} audio file(s):")
for p in AUDIO_FILES:
    print(" -", p.name + (" (+ reference .txt)" if p.with_suffix(".txt").exists() else ""))

## 8. Run the test

In [ ]:
import jiwer
SR = 16000

def split_chunks(audio, chunk_sec):
    size = int(chunk_sec * SR)
    if size <= 0 or len(audio) <= size:
        return [audio]
    return [audio[i:i + size] for i in range(0, len(audio), size)]

results, errors = [], []
for f in AUDIO_FILES:
    print("=" * 80)
    print("FILE:", f.name)
    rec = {"file": f.name, "status": "ok"}
    try:
        audio, _ = librosa.load(str(f), sr=SR, mono=True)
        if audio.size == 0:
            raise ValueError("audio decoded to 0 samples")
        dur = len(audio) / SR
        chunks = split_chunks(audio, CHUNK_SEC)
        t0 = time.time()
        text = " ".join(t for t in (model.transcribe(c) for c in chunks) if t)
        infer = time.time() - t0
        rec.update(duration_sec=round(dur, 2), chunks=len(chunks), inference_sec=round(infer, 2),
                   rtf=round(infer / dur, 3), text=text)
        print(f"duration {dur:.2f}s | chunks {len(chunks)} | inference {infer:.2f}s | RTF {infer/dur:.3f}")
        print("TRANSCRIPT:\n" + (text or "<empty>"))
        ref_file = f.with_suffix(".txt")
        if ref_file.exists():
            ref = ref_file.read_text(encoding="utf-8").strip()
            rec.update(wer=round(jiwer.wer(ref, text), 4), cer=round(jiwer.cer(ref, text), 4))
            print(f"WER {rec['wer']:.2%} | CER {rec['cer']:.2%}")
    except Exception as e:
        rec.update(status="error", error=f"{type(e).__name__}: {e}")
        errors.append((f.name, traceback.format_exc()))
        print("ERROR:", rec["error"])
    results.append(rec)

## 9. Summary

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(results)
display(df[[c for c in df.columns if c not in ("text", "segments")]])
ok = df[df.status == "ok"]
if len(ok):
    print(f"Model load {LOAD_SEC:.1f}s | total audio {ok.duration_sec.sum():.1f}s | "
          f"inference {ok.inference_sec.sum():.1f}s | overall RTF {ok.inference_sec.sum()/ok.duration_sec.sum():.3f}")
print(f"{len(ok)} ok, {len(errors)} failed")

## 10. Errors (full tracebacks)

In [ ]:
if not errors:
    print("No errors")
for name, tb in errors:
    print("=" * 80)
    print("FILE:", name)
    print(tb)

## 11. Save and download results

In [ ]:
from datetime import datetime
out_json = f"/content/whisper_small_bn_ctc_results_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump({"model": REPO_ID, "chunk_sec": CHUNK_SEC, "threads": torch.get_num_threads(), "load_sec": round(LOAD_SEC, 2),
               "results": results, "errors": [{"file": n, "traceback": t} for n, t in errors]},
              fh, ensure_ascii=False, indent=2)
print("Saved", out_json)
from google.colab import files
files.download(out_json)